# 06 — Label mining + candidates + non-leaky training set

Run the label-mining and dataset preparation scripts under scripts/HAB/preparation.


In [3]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

DATASETS_DIR = REPO_ROOT / "runs/datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
print("Existing runs/datasets CSVs:")
for p in sorted(DATASETS_DIR.glob("*.csv"))[:50]:
    print(" -", p.name)


REPO_ROOT: /Users/ameerfiras/REDNET-ML
Existing runs/datasets CSVs:
 - fusion_training_with_plants.csv
 - fusion_training_with_plants_backup.csv
 - fusion_training_with_plants_detfilled.csv
 - fusion_training_with_plants_splitlabels.csv
 - hab_candidates_review.csv
 - hab_train_mined.csv
 - hab_train_mined_aslabel.csv
 - hab_train_nonleaky.csv
 - plants_mined_candidates.csv
 - plants_mined_train.csv
 - plants_mined_train_with_detectors.csv


## 6.1 Print help for the key scripts (keeps notebook synced with your repo)


In [5]:
for s in [
    "mine_HAB_candidates.py",
    "repopulate_hab_hits.py",
    "make_hab_labels.py",
    "make_HAB_train_nonleaky.py",
    "merge_with_fusion_labels.py",
]:
    p = REPO_ROOT / "scripts/HAB/preparation" / s
    if p.exists():
        sh(f'python "{p}" --help', check=False)
    else:
        print("Missing:", p)



▶ python "/Users/ameerfiras/REDNET-ML/scripts/HAB/preparation/mine_HAB_candidates.py" --help
usage: mine_HAB_candidates.py [-h] --in_csv IN_CSV --out_train_csv
                              OUT_TRAIN_CSV --out_candidates_csv
                              OUT_CANDIDATES_CSV [--group_by GROUP_BY]
                              [--id_col ID_COL] [--drop_multimonth_scenes]
                              [--promote_heuristic] [--nflh_z NFLH_Z]
                              [--chl_z CHL_Z] [--nflh_q NFLH_Q]
                              [--chl_q CHL_Q] [--kd490_min KD490_MIN]

options:
  -h, --help            show this help message and exit
  --in_csv IN_CSV       Your merged chip_indices_clean_hab.csv after
                        append_modis
  --out_train_csv OUT_TRAIN_CSV
                        Output labeled CSV for training (with hab_label_final)
  --out_candidates_csv OUT_CANDIDATES_CSV
                        List of mined positives to review
  --group_by GROUP_BY
  --id_col ID_COL
 

## 6.2 Typical run sequence (uncomment to execute)

This stays accurate because the notebook above shows the real flags from `--help`.


In [ ]:

# sh("python scripts/HAB/preparation/mine_HAB_candidates.py ...")
# sh("python scripts/HAB/preparation/repopulate_hab_hits.py ...")
# sh("python scripts/HAB/preparation/make_hab_labels.py ...")
# sh("python scripts/HAB/preparation/make_HAB_train_nonleaky.py ...")


## 6.3 Inspect newest non-leaky table


In [6]:

import pandas as pd
train = newest_path("runs/datasets/hab_train_nonleaky*.csv")
print("Newest train table:", train)
if train:
    df = pd.read_csv(train)
    display(df.head(5))
    if "label" in df.columns:
        print(df["label"].value_counts(dropna=False).head(10))


Newest train table: /Users/ameerfiras/REDNET-ML/runs/datasets/hab_train_nonleaky.csv


,fai_mean,datetime,rednir_mean,valid_px,rednir_std,ndwi_mean,scene_id,ndwi_std,tile,fai_std,...,season,hab_label,__tag__,__src__,month_sin,month_cos,season_winter,season_spring,season_summer,season_autumn
0,0.044839,2017-09-09 06:56:19.027000+00:00,0.907081,16384,1.192093e-07,-0.157338,S2B_MSIL2A_20170909T065619_R063_T39QXG_2021020...,1.490116e-08,S2B_MSIL2A_20170909T065619_R063_T39QXG_2021020...,0.000000e+00,...,autumn,0,aerial_summer_2017,data/aerial_summer_2017/chip_indices_clean_hab...,-1.000000,-1.836970e-16,0,0,0,1
1,0.044839,2017-09-09 06:56:19.027000+00:00,0.907081,16384,1.192093e-07,-0.157338,S2B_MSIL2A_20170909T065619_R063_T39QXG_2021020...,1.490116e-08,S2B_MSIL2A_20170909T065619_R063_T39QXG_2021020...,0.000000e+00,...,autumn,0,aerial_summer_2017,data/aerial_summer_2017/chip_indices_clean_hab...,-1.000000,-1.836970e-16,0,0,0,1
2,0.054885,2017-08-25 06:56:21.026000+00:00,0.930767,16384,1.192093e-07,-0.153761,S2A_MSIL2A_20170825T065621_R063_T39QXG_2021020...,1.490116e-08,S2A_MSIL2A_20170825T065621_R063_T39QXG_2021020...,0.000000e+00,...,summer,0,aerial_summer_2017,data/aerial_summer_2017/chip_indices_clean_hab...,-0.866025,-5.000000e-01,0,0,1,0
3,0.054885,2017-08-25 06:56:21.026000+00:00,0.930767,16384,1.192093e-07,-0.153761,S2A_MSIL2A_20170825T065621_R063_T39QXG_2021020...,1.490116e-08,S2A_MSIL2A_20170825T065621_R063_T39QXG_2021020...,0.000000e+00,...,summer,0,aerial_summer_2017,data/aerial_summer_2017/chip_indices_clean_hab...,-0.866025,-5.000000e-01,0,0,1,0
4,0.047974,2017-08-20 06:56:19.027000+00:00,0.905486,16384,0.000000e+00,-0.165186,S2B_MSIL2A_20170820T065619_R063_T39QXG_2020101...,0.000000e+00,S2B_MSIL2A_20170820T065619_R063_T39QXG_2020101...,7.450581e-09,...,summer,0,aerial_summer_2017,data/aerial_summer_2017/chip_indices_clean_hab...,-0.866025,-5.000000e-01,0,0,1,0
